# Drug & cell-line catalog — harmonizing GDSC, CTRPv2, PRISM and DrugBank

Settles the two questions that had to be answered before any modelling could start: **which cell lines
does SCP542 share with each response dataset**, and **when are two datasets talking about the same
compound**.

**Its one persistent output is `data/drug/all_sources_drug_catalog.csv`** — the unified compound table
that drug selection reads (`compound_status`, `target`, `moa_or_pathway`). A from-scratch rebuild needs
this notebook, because `data/` is gitignored.

One-off harmonization and audit — **not part of the training pipeline**; nothing here is re-run per
experiment. The numbers it produces are written up in
[Step 01](../../../docs/steps/01-datasets-and-harmonization.md), which is where they are interpreted.

> ⚠️ **Not portable as written.** Most inputs are absolute paths under
> `/Users/selin/Desktop/OncoTox/...` rather than resolved through
> `scripts/layout.py`, so this will not run unmodified elsewhere.


In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.layout import DEFAULT_CTRP_SCORE, PipelinePaths

# Every input resolves through the path contract (13.08.2026). Until then this notebook carried 15
# absolute /Users/selin/Desktop/... paths and two `../data/...` relative ones, which made it the only
# notebook in the project that could not run on another machine -- or in a git worktree, since
# `<repo>/data/` is gitignored.
paths = PipelinePaths.build(None, 'hvg5000', DEFAULT_CTRP_SCORE)
DRUG_DIR = ROOT / 'data' / 'drug'          # repo-side, gitignored: DrugBank-derived, not redistributable
DRUG_DIR.mkdir(parents=True, exist_ok=True)
CTRP_META = paths.ctrp_dir                  # v20.meta.* only -- compound and cell-line metadata
PRISM_DIR = paths.prism_dir

print(f'CTRP metadata : {CTRP_META}')
print(f'PRISM         : {PRISM_DIR}')
print(f'response       : {paths.ctrp_response_csv.name}  (DrEval, score={paths.score})')
print(f'catalog out    : {DRUG_DIR}')

CTRP metadata : /Users/selin/Desktop/OncoTox/data/metadata/CTRPv2.0_2015_ctd2_ExpandedDataset
PRISM         : /Users/selin/Desktop/OncoTox/data/metadata/PRISM_REPURPOSED
response       : CTRPv2.csv  (DrEval, score=auc_cc)
catalog out    : /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/data/drug


## 1 · Cell-line rosters and the name normalization each one needs

Four sources, four naming conventions: SCP542 `Metadata.txt` and PRISM append a tissue suffix (split
on `_`), GDSC embeds hyphens (stripped here), CTRPv2's `v20.meta.per_cell_line.txt` is already plain.
Matching below is **trim + lowercase**.

> ⚠️ **This is not the production normalization.** The pipeline
> (`_normalize_cell_line` in `scripts/preprocessing/ctrp_to_h5ad.py`) also strips `-` from *every*
> source. Both rules give the same 190 CTRPv2 name matches, so the audit numbers hold — but the
> discrepancy is why joining on names rather than persistent identifiers is a review item.

> ### ⛔ GDSC was removed from this notebook on 13.08.2026 (Selin)
>
> **It was an artifact of a learnability analysis run once for a co-student, and the drug selection
> must not be done on GDSC.** The file itself is untouched
> (`data/GDSC2_fitted_dose_response_27Oct23.xlsx`) and Step 01 still records the dataset, its
> provenance and its undocumented `LN_IC50` processing — this removes the *arm of this catalog*, not
> the dataset.
>
> **It changes nothing in the pipeline, and that is checkable rather than asserted.**
> `scripts/annotation/drug_annotation.py` filters the catalog with `catalog[catalog.dataset ==
> "CTRPv2"]` and asserts the key type immediately after, so GDSC rows could never reach the panel.
> No script, and no other notebook, reads a GDSC row.
>
> What it does change is five descriptive numbers in
> [Step 01's overlap audit](../../../docs/steps/01-datasets-and-harmonization.md#overlap--coverage-audit-03042026),
> which are sourced from this notebook and are marked stale there until it re-runs.

In [2]:
scp542_scrna = pd.read_csv(paths.meta_file, sep="\t", skiprows=[1])
scp542_scrna["Cell_line"] = scp542_scrna["Cell_line"].str.split("_").str[0]
scp542_scrna.head()

/var/folders/6k/gr_1_h_97154rq71pm_q3jn40000gn/T/ipykernel_90905/2607909058.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  scp542_scrna = pd.read_csv(paths.meta_file, sep="\t", skiprows=[1])


,NAME,Cell_line,Pool_ID,Cancer_type,Genes_expressed,Discrete_cluster_minpts5_eps1.8,Discrete_cluster_minpts5_eps1.5,Discrete_cluster_minpts5_eps1.2,CNA_subclone,SkinPig_score,...,EMTII_score,EMTIII_score,IFNResp_score,p53Sen_score,EpiSen_score,StressResp_score,ProtMatu_score,ProtDegra_score,G1/S_score,G2/M_score
0,AAACCTGAGACATAAC-1-18,NCIH2126,18,Lung Cancer,4318,NaN,NaN,NaN,NaN,0.166,...,-0.935,-0.935,0.130,0.619,1.869,-0.004,0.805,0.896,0.424,-1.125
1,AACGTTGTCACCCGAG-1-18,NCIH2126,18,Lung Cancer,5200,NaN,NaN,NaN,NaN,-0.213,...,-1.027,-1.027,0.066,1.049,1.267,0.252,1.299,1.610,0.624,-0.048
2,AACTGGTAGACACGAC-1-18,NCIH2126,18,Lung Cancer,4004,NaN,NaN,NaN,NaN,-0.101,...,-0.677,-0.677,0.304,0.822,2.401,0.141,0.451,1.225,-0.795,0.064
3,AACTGGTAGGGCTTGA-1-18,NCIH2126,18,Lung Cancer,4295,NaN,NaN,NaN,NaN,-0.014,...,-0.735,-0.735,0.094,0.834,2.282,0.150,0.267,0.892,-0.238,1.118
4,AACTGGTAGTACTTGC-1-18,NCIH2126,18,Lung Cancer,4842,NaN,NaN,NaN,NaN,0.006,...,-0.821,-0.821,0.034,0.960,1.400,-0.012,-0.276,-0.428,0.267,0.791


In [3]:
# Reusable helper to compare SCP542 coverage in another cell-line dataset

def build_unique_with_norm(df, source_col, norm_col):
    return (
        df.assign(**{norm_col: df[source_col].astype(str).str.strip().str.lower()})
        .drop_duplicates(subset=norm_col)
        .copy()
    )


def report_scp542_missing(reference_unique, reference_norm_col, reference_label):
    print(f"Unique {reference_label} cell lines: {reference_unique[reference_norm_col].nunique()}")
    print(f"Unique scp542 cell lines: {scp542_unique['CCL_NAME_NORM'].nunique()}")

    ref_norm_set = set(reference_unique[reference_norm_col])
    missing_scp542 = scp542_unique.loc[
        ~scp542_unique["CCL_NAME_NORM"].isin(ref_norm_set)
    ].copy()

    print(f"\nUnique scp542 cell lines missing from {reference_label}: {len(missing_scp542)}")
    display(missing_scp542[["Cell_line"]].sort_values("Cell_line").head(50))
    return missing_scp542


scp542_unique = build_unique_with_norm(scp542_scrna, "Cell_line", "CCL_NAME_NORM")

In [4]:
ctrp = pd.read_csv(CTRP_META / "v20.meta.per_cell_line.txt", sep="\t")
ctrp.head()

,master_ccl_id,ccl_name,ccl_availability,ccle_primary_site,ccle_primary_hist,ccle_hist_subtype_1
0,1,697,ccle;public,haematopoietic_and_lymphoid_tissue,lymphoid_neoplasm,acute_lymphoblastic_B_cell_leukaemia
1,3,5637,ccle;public,urinary_tract,carcinoma,NaN
2,4,2313287,ccle;public,stomach,carcinoma,adenocarcinoma
3,5,1321N1,ccle,central_nervous_system,glioma,astrocytoma
4,6,143B,ccle,bone,osteosarcoma,NaN


In [5]:
# Compare unique cell lines (case-insensitive) between CTRP and SCP542
ctrp_unique = build_unique_with_norm(ctrp, "ccl_name", "CELL_LINE_NAME_NORM")
missing_scp542 = report_scp542_missing(ctrp_unique, "CELL_LINE_NAME_NORM", "CTRP")


Unique CTRP cell lines: 1107
Unique scp542 cell lines: 198

Unique scp542 cell lines missing from CTRP: 8


,Cell_line
52358,93VU
53188,JHU006
51641,JHU011
49471,JHU029
31688,NCIH2077
40844,NCIH292
48879,SCC47
52930,SCC90


In [6]:
prism = pd.read_csv(PRISM_DIR / "Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv")
prism["ccle_name"] = prism["ccle_name"].str.split("_").str[0]
prism.head()

,ccle_name,row_id,pool_id,culture,depmap_id,screen
0,KYSE510,ACH-000824::P107::PR500A::REP1M,P107,PR500A,ACH-000824,REP1M
1,HEC1A,ACH-000954::P107::PR500A::REP1M,P107,PR500A,ACH-000954,REP1M
2,MIAPACA2,ACH-000601::P101::PR500A::REP1M,P101,PR500A,ACH-000601,REP1M
3,SW620,ACH-000651::P108::PR500A::REP1M,P108,PR500A,ACH-000651,REP1M
4,SKHEP1,ACH-000361::P108::PR500A::REP1M,P108,PR500A,ACH-000361,REP1M


In [7]:
# Compare unique cell lines (case-insensitive) between PRISM and SCP542
prism_unique = build_unique_with_norm(prism, "ccle_name", "CELL_LINE_NAME_NORM")
missing_scp542 = report_scp542_missing(prism_unique, "CELL_LINE_NAME_NORM", "prism")


Unique prism cell lines: 915
Unique scp542 cell lines: 198

Unique scp542 cell lines missing from prism: 16


,Cell_line
52358,93VU
48519,CAKI2
44527,HCC366
53188,JHU006
51641,JHU011
49471,JHU029
30535,KPL1
4548,KPNSI9S
46212,NCIH2073
45863,OAW28


## 2 · Coverage summary across the three datasets

Cell-line overlap with SCP542 and compound counts, side by side.


In [8]:
# Summary table: SCP542 coverage across datasets

def normalized_cell_line_set(df, col, split_on_underscore=False):
    series = df[col].dropna().astype(str).str.strip().str.lower()
    if split_on_underscore:
        series = series.str.split("_").str[0]
    series = series[series != ""]
    return set(series.unique())


# Match preprocessing used earlier in the notebook for SCP542 names
scp542_set = normalized_cell_line_set(scp542_scrna, "Cell_line", split_on_underscore=True)
ctrp_set = normalized_cell_line_set(ctrp, "ccl_name")
prism_set = normalized_cell_line_set(prism, "ccle_name", split_on_underscore=True)

# Drug counts from dedicated compound metadata files
ctrp_compounds = pd.read_csv(
    CTRP_META / "v20.meta.per_compound.txt",
    sep="\t",
)
prism_compounds = pd.read_csv(
    PRISM_DIR / "Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv"
)
ctrp_drug_count = int(ctrp_compounds["master_cpd_id"].nunique())
prism_drug_count = int(prism_compounds["Drug.Name"].nunique())

summary_df = pd.DataFrame([
        {
        "Dataset": "CTRPv2",
        "Total Cell Lines": len(ctrp_set),
        "Overlap with SCP542": len(ctrp_set & scp542_set),
        "Missing Lines": len(scp542_set - ctrp_set),
        "Target Metric": "Viability / AUC",
        "Drug Count": ctrp_drug_count,
    },
    {
        "Dataset": "PRISM",
        "Total Cell Lines": len(prism_set),
        "Overlap with SCP542": len(prism_set & scp542_set),
        "Missing Lines": len(scp542_set - prism_set),
        "Target Metric": "Viability",
        "Drug Count": prism_drug_count,
    },
])

display(summary_df)

,Dataset,Total Cell Lines,Overlap with SCP542,Missing Lines,Target Metric,Drug Count
0,CTRPv2,1107,190,8,Viability / AUC,545
1,PRISM,915,182,16,Viability,6575


## 3 · The unified compound catalog — the notebook's one persistent output

Writes `all_sources_drug_catalog.csv` by mapping each source onto shared columns: GDSC from the
fitted dose-response table, CTRPv2 from `v20.meta.per_compound.txt`, PRISM from its Extended Primary
compound list. The CTRPv2 mapping is the one drug selection depends on —
`cpd_status` → `compound_status` (the FDA/clinical filter),
`gene_symbol_of_protein_target` → `target`, `target_or_activity_of_compound` → `moa_or_pathway`.

> ⚠️ **`target` is empty for some of the most informative compounds** (`paclitaxel`, `vincristine`,
> `ml210`, `ml162`) — they carry only `moa_or_pathway`. Any filter that requires a non-empty `target`
> silently drops them.


In [9]:
# Build a unified drug/compound catalog across CTRPv2 and PRISM.
# GDSC was dropped on 13.08.2026 (Selin) -- see the section heading above for why.


# CTRPv2 compounds from compound metadata
ctrp_compounds = pd.read_csv(
    CTRP_META / "v20.meta.per_compound.txt",
    sep="\t",
)
ctrp_drugs = (
    ctrp_compounds[
        [
            "master_cpd_id",
            "cpd_name",
            "broad_cpd_id",
            "cpd_status",
            "gene_symbol_of_protein_target",
            "target_or_activity_of_compound",
            "source_name",
            "source_catalog_id",
            "cpd_smiles",
            "inclusion_rationale",
            "top_test_conc_umol",
        ]
    ]
    .dropna(subset=["master_cpd_id"])
    .drop_duplicates(subset=["master_cpd_id"])
    .rename(
        columns={
            "master_cpd_id": "identifier",
            "cpd_name": "compound_name",
            "broad_cpd_id": "alt_identifier",
            "cpd_status": "compound_status",
            "gene_symbol_of_protein_target": "target",
            "target_or_activity_of_compound": "moa_or_pathway",
            "source_name": "vendor_or_source",
            "source_catalog_id": "vendor_catalog_id",
            "cpd_smiles": "smiles",
            "inclusion_rationale": "notes",
            "top_test_conc_umol": "top_test_conc_umol",
        }
    )
)
ctrp_drugs["dataset"] = "CTRPv2"
ctrp_drugs["identifier"] = ctrp_drugs["identifier"].astype(str)
ctrp_drugs["source_id_type"] = "master_cpd_id"

# PRISM compounds from repurposing compound list
prism_compounds = pd.read_csv(
    PRISM_DIR / "Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv"
)
prism_drugs = (
    prism_compounds[
        ["Drug.Name", "repurposing_target", "MOA", "IDs", "Synonyms", "screen", "dose"]
    ]
    .dropna(subset=["Drug.Name"])
    .drop_duplicates(subset=["Drug.Name"])
    .rename(
        columns={
            "Drug.Name": "identifier",
            "repurposing_target": "target",
            "MOA": "moa_or_pathway",
            "IDs": "alt_identifier",
            "Synonyms": "synonyms",
            "screen": "screen",
            "dose": "dose",
        }
    )
)
prism_drugs["compound_name"] = prism_drugs["identifier"]
prism_drugs["dataset"] = "PRISM"
prism_drugs["identifier"] = prism_drugs["identifier"].astype(str)
prism_drugs["source_id_type"] = "Drug.Name"

# Align columns and concatenate
all_drugs_df = pd.concat([ctrp_drugs, prism_drugs], ignore_index=True, sort=False)

# Keep identifier first, dataset second
base_cols = ["identifier", "dataset"]
other_cols = [c for c in all_drugs_df.columns if c not in base_cols]
all_drugs_df = all_drugs_df[base_cols + other_cols]

# Add a normalized name helper for future duplicate review
if "compound_name" in all_drugs_df.columns:
    all_drugs_df["compound_name_norm"] = (
        all_drugs_df["compound_name"].astype(str).str.strip().str.lower()
    )

output_csv = DRUG_DIR / "all_sources_drug_catalog.csv"
all_drugs_df.to_csv(output_csv, index=False)

print(f"Saved drug catalog to: {output_csv}")
print(f"Rows: {len(all_drugs_df)}")
print("Rows by dataset:")
print(all_drugs_df["dataset"].value_counts())

display(all_drugs_df.head())

Saved drug catalog to: /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/data/drug/all_sources_drug_catalog.csv
Rows: 7120
Rows by dataset:
dataset
PRISM     6575
CTRPv2     545
Name: count, dtype: int64


,identifier,dataset,compound_name,alt_identifier,compound_status,target,moa_or_pathway,vendor_or_source,vendor_catalog_id,smiles,notes,top_test_conc_umol,source_id_type,synonyms,screen,dose,compound_name_norm
0,1788,CTRPv2,CIL55,BRD-K46556387,probe,NaN,screening hit,Columbia University,NaN,CN(C)CCNC(=O)c1cc2CSc3cc(Cl)ccc3-c2s1,pilot-set,10.0,master_cpd_id,NaN,NaN,NaN,cil55
1,3588,CTRPv2,BRD4132,BRD-K86574132,probe,NaN,screening hit,ChemDiv Inc.,4998-1380,CC(C)N1C(=O)S\C(=C\c2ccc(Sc3nc4ccccc4[nH]3)o2)...,chromatin;pilot-set,160.0,master_cpd_id,NaN,NaN,NaN,brd4132
2,12877,CTRPv2,BRD6340,BRD-K35716340,probe,NaN,screening hit,ChemDiv Inc.,1988-0090,C(Cn1c2ccccc2c2ccccc12)c1nc2ccccc2[nH]1,chromatin;pilot-set,33.0,master_cpd_id,NaN,NaN,NaN,brd6340
3,17712,CTRPv2,ML006,BRD-K89692698,probe,S1PR3,agonist of sphingosine 1-phosphate receptor 3,Enamine Ltd.,Z1037336336,C1CN(CCO1)c1nnc(-c2ccccc2)c(n1)-c1ccccc1,pilot-set,530.0,master_cpd_id,NaN,NaN,NaN,ml006
4,18311,CTRPv2,Bax channel blocker,BRD-A18763547,probe,BAX,inhibitor of BAX-mediated mitochondrial cytoch...,Maybridge,RJC01737,OC(CN1CCNCC1)Cn1c2ccc(Br)cc2c2cc(Br)ccc12,pilot-set,33.0,master_cpd_id,NaN,NaN,NaN,bax channel blocker


## 4 · When are two datasets naming the same compound?

Two matching strategies, in increasing confidence: **normalized name**, and **canonical Broad BRD-ID**
(`BRD-[A-Z]\d{8}`, extracted by regex). BRD-IDs are per compound and stable across name synonyms and
salt forms, so they are the trustworthy link; free-text names are not.


In [10]:
# Analyze overlap strategies across datasets using the unified drug catalog
import re


def extract_canonical_brd(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r"BRD-[A-Z]\d{8}", str(value))
    return match.group(0) if match else pd.NA


# Work from the unified table created in the previous cell
drug_df = all_drugs_df.copy()
drug_df["compound_name_norm"] = drug_df["compound_name"].astype(str).str.strip().str.lower()
drug_df["brd_canonical"] = drug_df["alt_identifier"].map(extract_canonical_brd)

# Cache dataset-level tables/sets for reuse in this and later cells
dataset_frames = {
    ds: sub[
        ["identifier", "dataset", "compound_name", "compound_name_norm", "target", "moa_or_pathway", "brd_canonical"]
    ].drop_duplicates()
    for ds, sub in drug_df.groupby("dataset")
}
dataset_sets = {
    ds: set(frame["compound_name_norm"].dropna().unique()) for ds, frame in dataset_frames.items()
}
datasets = sorted(dataset_frames.keys())

# Pairwise overlap summary (name-based and BRD-based)
summary_rows = []
for i, ds1 in enumerate(datasets):
    for ds2 in datasets[i + 1:]:
        a = dataset_frames[ds1]
        b = dataset_frames[ds2]
        summary_rows.append(
            {
                "dataset_a": ds1,
                "dataset_b": ds2,
                "name_overlap_count": len(dataset_sets[ds1] & dataset_sets[ds2]),
                "brd_overlap_count": len(set(a["brd_canonical"].dropna()) & set(b["brd_canonical"].dropna())),
            }
        )

overlap_summary_df = pd.DataFrame(summary_rows)
print("Pairwise overlap summary:")
display(overlap_summary_df)

# Build candidate overlap pairs by exact normalized name
name_candidate_frames = []
for i, ds1 in enumerate(datasets):
    for ds2 in datasets[i + 1:]:
        a = dataset_frames[ds1]
        b = dataset_frames[ds2]

        merged = a.merge(b, on="compound_name_norm", suffixes=("_a", "_b"), how="inner")
        if merged.empty:
            continue

        out = pd.DataFrame(
            {
                "dataset_a": merged["dataset_a"],
                "identifier_a": merged["identifier_a"],
                "compound_name_a": merged["compound_name_a"],
                "dataset_b": merged["dataset_b"],
                "identifier_b": merged["identifier_b"],
                "compound_name_b": merged["compound_name_b"],
                "compound_name_norm": merged["compound_name_norm"],
                "brd_canonical": merged["brd_canonical_a"].fillna(merged["brd_canonical_b"]),
                "match_method": "name_exact",
                "target_a": merged["target_a"],
                "target_b": merged["target_b"],
                "moa_or_pathway_a": merged["moa_or_pathway_a"],
                "moa_or_pathway_b": merged["moa_or_pathway_b"],
            }
        )
        name_candidate_frames.append(out)

name_candidates_df = (
    pd.concat(name_candidate_frames, ignore_index=True)
    if name_candidate_frames
    else pd.DataFrame()
)

# Build high-confidence CTRPv2 <-> PRISM matches by canonical BRD ID
ctrp_subset = dataset_frames.get("CTRPv2", pd.DataFrame())
prism_subset = dataset_frames.get("PRISM", pd.DataFrame())

brd_merged = ctrp_subset.merge(prism_subset, on="brd_canonical", suffixes=("_a", "_b"), how="inner")
brd_candidates_df = pd.DataFrame(
    {
        "dataset_a": brd_merged["dataset_a"],
        "identifier_a": brd_merged["identifier_a"],
        "compound_name_a": brd_merged["compound_name_a"],
        "dataset_b": brd_merged["dataset_b"],
        "identifier_b": brd_merged["identifier_b"],
        "compound_name_b": brd_merged["compound_name_b"],
        "compound_name_norm": brd_merged["compound_name_norm_a"],
        "brd_canonical": brd_merged["brd_canonical"],
        "match_method": "brd_exact",
        "target_a": brd_merged["target_a"],
        "target_b": brd_merged["target_b"],
        "moa_or_pathway_a": brd_merged["moa_or_pathway_a"],
        "moa_or_pathway_b": brd_merged["moa_or_pathway_b"],
    }
)

# Combine and de-duplicate candidate edges
candidates_df = pd.concat([name_candidates_df, brd_candidates_df], ignore_index=True)
candidates_df = candidates_df.drop_duplicates(
    subset=["dataset_a", "identifier_a", "dataset_b", "identifier_b", "match_method"]
)

print("\nCandidate overlap pairs by match method:")
print(candidates_df["match_method"].value_counts())

# Save for manual review/dedup curation
candidate_output = DRUG_DIR / "drug_overlap_candidates.csv"
summary_output = DRUG_DIR / "drug_overlap_summary.csv"
candidates_df.to_csv(candidate_output, index=False)
overlap_summary_df.to_csv(summary_output, index=False)

print(f"Saved candidate overlap pairs to: {candidate_output}")
print(f"Saved overlap summary to: {summary_output}")

display(candidates_df.head(20))

Pairwise overlap summary:


,dataset_a,dataset_b,name_overlap_count,brd_overlap_count
0,CTRPv2,PRISM,218,243



Candidate overlap pairs by match method:
match_method
brd_exact     243
name_exact    218
Name: count, dtype: int64
Saved candidate overlap pairs to: /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/data/drug/drug_overlap_candidates.csv
Saved overlap summary to: /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/data/drug/drug_overlap_summary.csv


,dataset_a,identifier_a,compound_name_a,dataset_b,identifier_b,compound_name_b,compound_name_norm,brd_canonical,match_method,target_a,target_b,moa_or_pathway_a,moa_or_pathway_b
0,CTRPv2,19153,BRD9876,PRISM,BRD9876,BRD9876,brd9876,BRD-K89329876,name_exact,NaN,KIF11,screening hit,KINESIN INHIBITOR
1,CTRPv2,23151,tretinoin,PRISM,TRETINOIN,TRETINOIN,tretinoin,BRD-K64634304,name_exact,RARA;RARB;RARG,"ALDH1A1, ALDH1A2, GPRC5A, NR0B1, NR2C2, PPARD,...",agonist of retinoid acid receptors,"RETINOID RECEPTOR AGONIST, RETINOID RECEPTOR L..."
2,CTRPv2,24173,phloretin,PRISM,PHLORETIN,PHLORETIN,phloretin,BRD-K15563106,name_exact,SLC5A1,"AQP9, CLCN3, SLC23A1",natural product; inhibitor of glucose uptake,SODIUM/GLUCOSE COTRANSPORTER INHIBITOR
3,CTRPv2,24197,thalidomide,PRISM,THALIDOMIDE,THALIDOMIDE,thalidomide,BRD-A93255169,name_exact,CRBN,"CRBN, FGFR2, NFKB1, ORM1, ORM2, PTGS2, TNF",immunomodulatory drug; binder of cereblon,TUMOR NECROSIS FACTOR PRODUCTION INHIBITOR
4,CTRPv2,25036,gossypol,PRISM,GOSSYPOL,GOSSYPOL,gossypol,BRD-K19295594,name_exact,BCL2;BCL2L1;LDHA;LDHB;LDHC,BCL2,inhibitor of lactate dehydrogenase; inhibitor ...,"BCL INHIBITOR, MCL1 INHIBITOR"
5,CTRPv2,25334,chlorambucil,PRISM,CHLORAMBUCIL,CHLORAMBUCIL,chlorambucil,BRD-K29458283,name_exact,NaN,NaN,DNA alkylator,DNA INHIBITOR
6,CTRPv2,25393,isoliquiritigenin,PRISM,ISOLIQUIRITIGENIN,ISOLIQUIRITIGENIN,isoliquiritigenin,BRD-K33583600,name_exact,NaN,GABBR1,natural product,GUANYLATE CYCLASE ACTIVATOR
7,CTRPv2,26870,cimetidine,PRISM,CIMETIDINE,CIMETIDINE,cimetidine,BRD-K34157611,name_exact,HRH2,"HRH2, SLC29A4, SLC47A1, SLC47A2",inhibitor of histidine receptor H2,HISTAMINE RECEPTOR ANTAGONIST
8,CTRPv2,26874,azacitidine,PRISM,AZACITIDINE,AZACITIDINE,azacitidine,BRD-K03406345,name_exact,DNMT1,"DNMT1, DNMT3A",inhibitor of DNA methyltransferase,DNA METHYLTRANSFERASE INHIBITOR
9,CTRPv2,26914,trifluoperazine,PRISM,TRIFLUOPERAZINE,TRIFLUOPERAZINE,trifluoperazine,BRD-K89732114,name_exact,DRD2,"ADRA1A, CALM1, CALY, DRD2, DRD4, HRH1, HTR2A, ...",antagonist of dopamine receptor D2,DOPAMINE RECEPTOR ANTAGONIST


In [11]:
# Summary stats from overlap artifacts generated in the previous cell
from collections import Counter

# Reuse prepared objects from previous cell; rebuild only if needed
if "dataset_sets" not in globals():
    dataset_sets = {
        ds: set(sub["compound_name_norm"].dropna().unique())
        for ds, sub in drug_df.groupby("dataset")
    }

if "overlap_summary_df" not in globals():
    summary_rows = []
    datasets = sorted(dataset_sets.keys())
    for i, ds1 in enumerate(datasets):
        for ds2 in datasets[i + 1:]:
            set_a = dataset_sets[ds1]
            set_b = dataset_sets[ds2]
            summary_rows.append(
                {
                    "dataset_a": ds1,
                    "dataset_b": ds2,
                    "name_overlap_count": len(set_a & set_b),
                }
            )
    overlap_summary_df = pd.DataFrame(summary_rows)

# Unique counts
unique_per_dataset = (
    pd.Series({ds: len(compounds) for ds, compounds in dataset_sets.items()})
    .sort_values(ascending=False)
    .rename_axis("dataset")
    .reset_index(name="unique_compound_count")
)
all_compounds = set().union(*dataset_sets.values())
overall_unique_compounds = len(all_compounds)

print("Unique compounds per dataset:")
display(unique_per_dataset)
print(f"Overall unique compounds across all datasets (union): {overall_unique_compounds}")

# Add percentages to existing pairwise overlap counts
pairwise_overlap_df = overlap_summary_df.copy()
if "overlap_count" in pairwise_overlap_df.columns and "name_overlap_count" not in pairwise_overlap_df.columns:
    pairwise_overlap_df = pairwise_overlap_df.rename(columns={"overlap_count": "name_overlap_count"})

pairwise_overlap_df["set_size_a"] = pairwise_overlap_df["dataset_a"].map(lambda ds: len(dataset_sets[ds]))
pairwise_overlap_df["set_size_b"] = pairwise_overlap_df["dataset_b"].map(lambda ds: len(dataset_sets[ds]))
pairwise_overlap_df["union_size"] = pairwise_overlap_df.apply(
    lambda r: len(dataset_sets[r["dataset_a"]] | dataset_sets[r["dataset_b"]]), axis=1
)
pairwise_overlap_df["pct_of_a"] = 100 * pairwise_overlap_df["name_overlap_count"] / pairwise_overlap_df["set_size_a"]
pairwise_overlap_df["pct_of_b"] = 100 * pairwise_overlap_df["name_overlap_count"] / pairwise_overlap_df["set_size_b"]
pairwise_overlap_df["jaccard_pct"] = 100 * pairwise_overlap_df["name_overlap_count"] / pairwise_overlap_df["union_size"]

print("\nPairwise overlap percentages (name-based):")
display(pairwise_overlap_df[["dataset_a", "dataset_b", "name_overlap_count", "pct_of_a", "pct_of_b", "jaccard_pct"]])

# Membership distribution: in how many datasets each compound appears
membership_counts = Counter(
    sum(comp in compounds for compounds in dataset_sets.values())
    for comp in all_compounds
)
membership_summary_df = (
    pd.Series(membership_counts)
    .sort_index()
    .rename_axis("present_in_n_datasets")
    .reset_index(name="compound_count")
)
membership_summary_df["pct_of_overall_unique"] = (
    100 * membership_summary_df["compound_count"] / overall_unique_compounds
)

print("\nDistribution of compounds by number of datasets they appear in:")
display(membership_summary_df)

# Exact dataset-combination breakdown
combo_counts = Counter(
    " | ".join(sorted(ds for ds, compounds in dataset_sets.items() if comp in compounds))
    for comp in all_compounds
)
combo_breakdown_df = (
    pd.Series(combo_counts)
    .sort_values(ascending=False)
    .rename_axis("dataset_combination")
    .reset_index(name="compound_count")
)
combo_breakdown_df["pct_of_overall_unique"] = (
    100 * combo_breakdown_df["compound_count"] / overall_unique_compounds
)

print("\nExact dataset-combination breakdown:")
display(combo_breakdown_df)


Unique compounds per dataset:


,dataset,unique_compound_count
0,PRISM,6575
1,CTRPv2,545


Overall unique compounds across all datasets (union): 6902

Pairwise overlap percentages (name-based):


,dataset_a,dataset_b,name_overlap_count,pct_of_a,pct_of_b,jaccard_pct
0,CTRPv2,PRISM,218,40.0,3.315589,3.158505



Distribution of compounds by number of datasets they appear in:


,present_in_n_datasets,compound_count,pct_of_overall_unique
0,1,6684,96.841495
1,2,218,3.158505



Exact dataset-combination breakdown:


,dataset_combination,compound_count,pct_of_overall_unique
0,PRISM,6357,92.103738
1,CTRPv2,327,4.737757
2,CTRPv2 | PRISM,218,3.158505


## 5 · DrugBank cross-reference

Matches the catalog against the DrugBank XML on a hard-normalized name (all non-alphanumerics
stripped, so `5-fluorouracil` = `5 fluorouracil`), expanded with synonyms where a source provides
them. This is what makes FDA / clinical-status filtering possible downstream.


In [12]:
drugbank = pd.read_xml(paths.drugbank_file)
drugbank.head()

,type,created,updated,drugbank-id,name,description,cas-number,unii,state,groups,...,snp-adverse-drug-reactions,targets,enzymes,carriers,transporters,fda-label,msds,average-mass,monoisotopic-mass,calculated-properties
0,biotech,2005-06-13,2026-02-02,BIOD00024,Lepirudin,Lepirudin is a recombinant hirudin formed by 6...,138068-37-8,Y43GF64R34,solid,\n,...,None,\n,None,None,None,None,None,NaN,NaN,None
1,biotech,2005-06-13,2026-03-05,BIOD00071,Cetuximab,Cetuximab is a recombinant chimeric human/mous...,205923-56-4,PQX0D8J21J,liquid,\n,...,None,\n,None,None,None,//s3-us-west-2.amazonaws.com/drugbank/fda_labe...,//s3-us-west-2.amazonaws.com/drugbank/msds/DB0...,NaN,NaN,None
2,biotech,2005-06-13,2026-03-05,BIOD00001,Dornase alfa,Dornase alfa is a biosynthetic form of human d...,143831-71-4,953A26OA1Y,liquid,\n,...,None,\n,None,None,None,//s3-us-west-2.amazonaws.com/drugbank/fda_labe...,//s3-us-west-2.amazonaws.com/drugbank/msds/DB0...,NaN,NaN,None
3,biotech,2005-06-13,2026-03-05,BIOD00084,Denileukin diftitox,Denileukin diftitox is an IL2-receptor-directe...,173146-27-5,25E79B5CTM,liquid,\n,...,None,\n,None,None,None,None,None,NaN,NaN,None
4,biotech,2005-06-13,2026-03-05,BIOD00052,Etanercept,Dimeric fusion protein consisting of the extra...,185243-69-0,OP401G7OJC,liquid,\n,...,None,\n,None,None,None,None,None,NaN,NaN,None


In [13]:
# Overlap between each dataset and DrugBank
import re


def normalize_drug_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    # Remove punctuation/spacing differences (e.g., "5-fluorouracil" vs "5 fluorouracil")
    text = re.sub(r"[^a-z0-9]+", "", text)
    return text


# Ensure drug catalog and DrugBank are available
if "all_drugs_df" not in globals():
    all_drugs_df = pd.read_csv(DRUG_DIR / "all_sources_drug_catalog.csv")
if "drugbank" not in globals():
    drugbank = pd.read_xml(paths.drugbank_file)

# Prepare DrugBank keys
drugbank_ref = drugbank[["drugbank-id", "name", "groups", "cas-number", "unii"]].copy()
drugbank_ref["drugbank_name_norm"] = drugbank_ref["name"].map(normalize_drug_text)
drugbank_ref = drugbank_ref[drugbank_ref["drugbank_name_norm"] != ""].drop_duplicates(
    subset=["drugbank_name_norm"]
)
drugbank_name_set = set(drugbank_ref["drugbank_name_norm"])

# Prepare source compound keys (name + optional synonym expansion)
catalog = all_drugs_df.copy()
catalog["compound_name_norm"] = catalog["compound_name"].map(normalize_drug_text)
catalog = catalog[catalog["compound_name_norm"] != ""]

# Optional synonym tokens from PRISM/other sources if present
synonym_rows = []
if "synonyms" in catalog.columns:
    syn_source = catalog.dropna(subset=["synonyms"])[["identifier", "dataset", "compound_name", "synonyms"]].copy()
    for _, row in syn_source.iterrows():
        parts = re.split(r"[|;,]", str(row["synonyms"]))
        for syn in parts:
            syn_norm = normalize_drug_text(syn)
            if syn_norm:
                synonym_rows.append(
                    {
                        "identifier": row["identifier"],
                        "dataset": row["dataset"],
                        "compound_name": row["compound_name"],
                        "compound_name_norm": syn_norm,
                        "match_basis": "synonym",
                    }
                )

name_rows = catalog[["identifier", "dataset", "compound_name", "compound_name_norm"]].copy()
name_rows["match_basis"] = "name"

expanded_catalog = pd.concat(
    [name_rows, pd.DataFrame(synonym_rows)], ignore_index=True, sort=False
).drop_duplicates(subset=["dataset", "identifier", "compound_name_norm", "match_basis"])

# Match to DrugBank by normalized name key
matched = expanded_catalog[expanded_catalog["compound_name_norm"].isin(drugbank_name_set)].copy()
unmatched = expanded_catalog[~expanded_catalog["compound_name_norm"].isin(drugbank_name_set)].copy()

# Keep one representative match per dataset+identifier
matched_unique = matched.sort_values(by=["dataset", "identifier", "match_basis"]).drop_duplicates(
    subset=["dataset", "identifier"], keep="first"
)

# Attach DrugBank metadata to matched entries
matched_with_db = matched_unique.merge(
    drugbank_ref,
    left_on="compound_name_norm",
    right_on="drugbank_name_norm",
    how="left",
)

# Summary per dataset
total_unique_per_dataset = (
    catalog.groupby("dataset")["identifier"].nunique().rename("dataset_unique_compounds")
)
matched_unique_per_dataset = (
    matched_unique.groupby("dataset")["identifier"].nunique().rename("matched_in_drugbank")
)

overlap_vs_drugbank_df = (
    pd.concat([total_unique_per_dataset, matched_unique_per_dataset], axis=1)
    .fillna(0)
    .reset_index()
)
overlap_vs_drugbank_df["matched_in_drugbank"] = overlap_vs_drugbank_df[
    "matched_in_drugbank"
].astype(int)
overlap_vs_drugbank_df["pct_dataset_in_drugbank"] = (
    100
    * overlap_vs_drugbank_df["matched_in_drugbank"]
    / overlap_vs_drugbank_df["dataset_unique_compounds"]
)

# Overall stats
overall_unique = int(catalog["identifier"].nunique())
overall_matched = int(matched_unique["identifier"].nunique())
overall_pct = 100 * overall_matched / overall_unique if overall_unique else 0

print("Overlap with DrugBank by dataset:")
display(overlap_vs_drugbank_df.sort_values("dataset"))
print(
    f"Overall matched unique compounds: {overall_matched}/{overall_unique} ({overall_pct:.2f}%)"
)

# Save review tables
matched_out = DRUG_DIR / "drugbank_overlap_matches.csv"
unmatched_out = DRUG_DIR / "drugbank_overlap_unmatched.csv"
summary_out = DRUG_DIR / "drugbank_overlap_summary.csv"

matched_with_db.to_csv(matched_out, index=False)
unmatched.drop_duplicates(subset=["dataset", "identifier"]).to_csv(unmatched_out, index=False)
overlap_vs_drugbank_df.to_csv(summary_out, index=False)

print(f"Saved matched table: {matched_out}")
print(f"Saved unmatched table: {unmatched_out}")
print(f"Saved summary table: {summary_out}")

display(matched_with_db[["dataset", "identifier", "compound_name", "match_basis", "drugbank-id", "name", "groups"]].head(20))

Overlap with DrugBank by dataset:


,dataset,dataset_unique_compounds,matched_in_drugbank,pct_dataset_in_drugbank
0,CTRPv2,545,173,31.743119
1,PRISM,6575,3483,52.973384


Overall matched unique compounds: 3656/7120 (51.35%)
Saved matched table: /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/data/drug/drugbank_overlap_matches.csv
Saved unmatched table: /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/data/drug/drugbank_overlap_unmatched.csv
Saved summary table: /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/data/drug/drugbank_overlap_summary.csv


,dataset,identifier,compound_name,match_basis,drugbank-id,name,groups
0,CTRPv2,23151,tretinoin,name,APRD00362,Tretinoin,\n
1,CTRPv2,23256,betulinic acid,name,DB05910,Betulinic Acid,\n
2,CTRPv2,24173,phloretin,name,DB07810,Phloretin,\n
3,CTRPv2,24197,thalidomide,name,APRD01251,Thalidomide,\n
4,CTRPv2,25036,gossypol,name,DB13044,Gossypol,\n
5,CTRPv2,25334,chlorambucil,name,APRD00115,Chlorambucil,\n
6,CTRPv2,25344,fluorouracil,name,EXPT03204,Fluorouracil,\n
7,CTRPv2,25393,isoliquiritigenin,name,EXPT01707,Isoliquiritigenin,\n
8,CTRPv2,26870,cimetidine,name,APRD00568,Cimetidine,\n
9,CTRPv2,26874,azacitidine,name,APRD00809,Azacitidine,\n


## 6 · How much response data exists for the overlapping lines

Non-null target values for the SCP542-overlapping lines, per dataset: GDSC `LN_IC50`, CTRPv2
`cpd_avg_pv` (joined `experiment_id` → `ccl_name`), PRISM's Extended Primary matrix. This is the
comparison that made **CTRPv2 the starting database** — see [Step 01](../../../docs/steps/01-datasets-and-harmonization.md).


In [14]:
# Target-value coverage for overlapping SCP542 cell lines vs each dataset's own cell lines
# Assumptions (revised 13.08.2026):
# - CTRPv2 target metric: the score this project trains on, from DrEval's reprocessed CTRPv2.
#   This USED TO read v20.data.per_cpd_post_qc.txt's `cpd_avg_pv` -- CTRPv2's own 2015 distribution,
#   which stopped being the response source on 11.08.2026. Counting coverage from a table the
#   pipeline no longer trains on described a dataset this project does not use.
# - GDSC was dropped entirely (Selin, 13.08.2026): see §1.
# - PRISM target metric: values in Extended Primary Data Matrix (per compound x depmap cell line)

# SCP542 normalized cell-line names used for overlap checks
scp542_overlap_set = set(
    scp542_scrna["Cell_line"].dropna().astype(str).str.strip().str.lower().unique()
)

# -----------------------------------------
# CTRPv2: DrEval's reprocessed response table, keyed by ccl_name
# -----------------------------------------
# One file replaces the three-way v20 join (values -> experiment -> cell line): CTRPv2.csv already
# carries the cell-line name beside the response, so the joins that reconstructed it are no longer
# needed. The metric is `paths.score`, i.e. what the model is actually trained against.
# The project's score NAMES are not the file's column names: `auc_cc` is `AUC_curvecurator` in
# DrEval's table. ctrp_to_h5ad.SCORE_COLUMNS is the one mapping between them, imported rather than
# restated so this cannot drift from what the pipeline actually trains on (fixed 13.08.2026, Gate 5).
from scripts.preprocessing.ctrp_to_h5ad import SCORE_COLUMNS

SCORE_COL = SCORE_COLUMNS[paths.score]
ctrp_eval = pd.read_csv(paths.ctrp_response_csv, usecols=["ccl_name", SCORE_COL])
ctrp_eval["cell_line_norm"] = (
    ctrp_eval["ccl_name"].astype(str).str.strip().str.lower()
)

ctrp_non_null_own = int(ctrp_eval[SCORE_COL].notna().sum())
ctrp_overlap_mask = ctrp_eval["cell_line_norm"].isin(scp542_overlap_set)
ctrp_total_overlap_rows = int(ctrp_overlap_mask.sum())
ctrp_non_null_overlap = int(
    ctrp_eval.loc[ctrp_overlap_mask, SCORE_COL].notna().sum()
)

# ---------------------------------------------------------
# PRISM: matrix values across depmap columns (non-null cells)
# ---------------------------------------------------------
prism_matrix = pd.read_csv(
    PRISM_DIR / "Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv"
)
prism_cl_meta = pd.read_csv(
    PRISM_DIR / "Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv"
)

# Map SCP542 names -> DepMap IDs through PRISM cell-line metadata
prism_cl_meta["base_ccle_name"] = (
    prism_cl_meta["ccle_name"].astype(str).str.split("_").str[0].str.strip().str.lower()
)
overlap_depmap_ids = set(
    prism_cl_meta.loc[
        prism_cl_meta["base_ccle_name"].isin(scp542_overlap_set), "depmap_id"
    ]
    .dropna()
    .astype(str)
)

value_cols = [c for c in prism_matrix.columns if c != "Unnamed: 0"]
overlap_value_cols = [c for c in value_cols if c in overlap_depmap_ids]

prism_non_null_own = int(prism_matrix[value_cols].notna().sum().sum())
prism_total_overlap_rows = int(prism_matrix[overlap_value_cols].size)
prism_non_null_overlap = int(prism_matrix[overlap_value_cols].notna().sum().sum())

# ---------------------
# Summary table
# ---------------------
target_coverage_df = pd.DataFrame(
    [
                {
            "dataset": "CTRPv2",
            "target_metric": SCORE_COL,
            "non_null_rows_own_cell_lines": ctrp_non_null_own,
            "rows_in_overlap_subset": ctrp_total_overlap_rows,
            "non_null_rows_overlap_scp542": ctrp_non_null_overlap,
        },
        {
            "dataset": "PRISM",
            "target_metric": "extended primary matrix value (viability proxy)",
            "non_null_rows_own_cell_lines": prism_non_null_own,
            "rows_in_overlap_subset": prism_total_overlap_rows,
            "non_null_rows_overlap_scp542": prism_non_null_overlap,
        },
    ]
)
target_coverage_df["pct_overlap_vs_own"] = (
    100
    * target_coverage_df["non_null_rows_overlap_scp542"]
    / target_coverage_df["non_null_rows_own_cell_lines"]
)

# This is the "against SCP overlap subset" percentage: non-null within overlap-only rows
# (i.e., completeness of response values among overlap rows)
target_coverage_df["pct_non_null_within_overlap_subset"] = (
    100
    * target_coverage_df["non_null_rows_overlap_scp542"]
    / target_coverage_df["rows_in_overlap_subset"]
)

display(target_coverage_df)

,dataset,target_metric,non_null_rows_own_cell_lines,rows_in_overlap_subset,non_null_rows_overlap_scp542,pct_overlap_vs_own,pct_non_null_within_overlap_subset
0,CTRPv2,AUC_curvecurator,395024,84350,84350,21.353133,100.000000
1,PRISM,extended primary matrix value (viability proxy),4213048,1235780,1210432,28.730553,97.948826
